In [2]:
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

In [32]:
df = pd.read_csv(os.getenv("ONLINE_RETAIL2_PATH"), encoding='ISO-8859-1')

# Estructura y tipos
print(df.info())
print("\nPorcentaje de nulos por columna\n", df.isnull().mean())
print("\nNumero de duplicados en el DF: ", df.duplicated().sum())
print("\nPorcentaje de duplicados en Invoice\n", df['Invoice'].duplicated().mean())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB
None

Porcentaje de nulos por columna
 Invoice        0.000000
StockCode      0.000000
Description    0.004105
Quantity       0.000000
InvoiceDate    0.000000
Price          0.000000
Customer ID    0.227669
Country        0.000000
dtype: float64

Numero de duplicados en el DF:  34335

Porcentaje de duplicados en Invoice
 0.949756926129715


Customer_ID tiene 22.7% de nulos. <br> <br>
Invoice tiene un alto porcentaje de duplicados (94.9%) por lo que es muy probable que los repetidos sean otros items del mismo Invoice


In [42]:
df[df.duplicated(keep=False)].sort_values('Invoice').head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
390,489517,84951A,S/4 PISTACHIO LOVEBIRD COASTERS,1,2009-12-01 11:34:00,2.55,16329.0,United Kingdom
388,489517,84951A,S/4 PISTACHIO LOVEBIRD COASTERS,1,2009-12-01 11:34:00,2.55,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom


HALLAZGO: Las filas duplicadas representan el mismo registro capturado dos veces (ejemplo: indice 362 y 385). Candidatas a deduplicacion en capa Silver.

In [21]:
# Calidad de filas
print("\nNumero de filas con Quantity negativo: ", (df["Quantity"] < 0).sum())
print("\nNumero de filas con Price negativo: ", (df["Price"] <= 0).sum())
print("\nNumero de cancelaciones: ", (df['Invoice'].astype(str).str.startswith('C')).sum())

neg_qty_cancelled = df[(df['Quantity'] < 0) & (df['Invoice'].str.startswith('C'))]
print(neg_qty_cancelled)

neg_qty_no_cancel = df[(df['Quantity'] < 0) & (~df['Invoice'].astype(str).str.startswith('C'))]
print(neg_qty_no_cancel)


Numero de filas con Quantity negativo:  22950

Numero de filas con Price negativo:  6207

Numero de cancelaciones:  19494
         Invoice StockCode                       Description  Quantity  \
178      C489449     22087          PAPER BUNTING WHITE LACE       -12   
179      C489449    85206A      CREAM FELT EASTER EGG BASKET        -6   
180      C489449     21895     POTTING SHED SOW 'N' GROW SET        -4   
181      C489449     21896                POTTING SHED TWINE        -6   
182      C489449     22083        PAPER CHAIN KIT RETRO SPOT       -12   
...          ...       ...                               ...       ...   
1065910  C581490     23144   ZINC T-LIGHT HOLDER STARS SMALL       -11   
1067002  C581499         M                            Manual        -1   
1067176  C581568     21258        VICTORIAN SEWING BOX LARGE        -5   
1067177  C581569     84978  HANGING HEART JAR T-LIGHT HOLDER        -1   
1067178  C581569     20979     36 PENCILS TUBE RED RETROSPOT   

HALLAZGO: Las ordenes con Quantity , con StockCode e Invoice valido sin haber sido canceladas muestran corrupcion en los campos de Description y Customer ID. Siendo un total de 3,457 filas corruptas (ajustes internos de inventario, mermas, productos dañados, etc).<br> <br>
Ademas estas filas presentan Customer ID nulos, lo que explica el procentaje de 22.7% nulos en la columna Customer ID. <br> <br>
A diferencia de ordenes canceladas con Quantity negativa que si muestran campos de Description, Customer ID y StockCode validos.

In [14]:
# StockCode sospechosos
print(df['StockCode'].value_counts().to_string())

StockCode
85123A          5829
22423           4424
85099B          4216
21212           3318
20725           3259
84879           2960
47566           2768
21232           2747
22197           2549
22383           2540
20727           2529
21931           2434
22386           2347
22469           2325
22411           2297
84991           2271
22382           2251
22384           2230
21080           2224
22086           2217
20914           2213
21034           2199
22139           2196
21754           2192
20728           2185
22138           2154
22457           2152
20724           2138
POST            2122
22470           2117
82482           2116
82494L          2108
21977           2092
20726           2050
21790           1988
85099C          1952
22178           1934
21181           1920
85099F          1916
84946           1904
21733           1903
22077           1858
22112           1847
22111           1846
22355           1832
22666           1810
21755           1798
206

StockCode sospechosos: <br>
POST = 2122 <br>
DOT = 1446 <br>
M = 1421 <br>
BANK CHARGES = 102 <br>

In [15]:
print(df['Country'].unique())

['United Kingdom' 'France' 'USA' 'Belgium' 'Australia' 'EIRE' 'Germany'
 'Portugal' 'Japan' 'Denmark' 'Nigeria' 'Netherlands' 'Poland' 'Spain'
 'Channel Islands' 'Italy' 'Cyprus' 'Greece' 'Norway' 'Austria' 'Sweden'
 'United Arab Emirates' 'Finland' 'Switzerland' 'Unspecified' 'Malta'
 'Bahrain' 'RSA' 'Bermuda' 'Hong Kong' 'Singapore' 'Thailand' 'Israel'
 'Lithuania' 'West Indies' 'Lebanon' 'Korea' 'Brazil' 'Canada' 'Iceland'
 'Saudi Arabia' 'Czech Republic' 'European Community']


Pasies sospechosos: <br>
EIRE - RSA

In [17]:
print(df['InvoiceDate'].min(), " -> ", df['InvoiceDate'].max())

2009-12-01 07:45:00  ->  2011-12-09 12:50:00


In [66]:
print(df.nlargest(5, 'Price'))
print(df.nsmallest(5, 'Price'))

         Invoice     StockCode   Description  Quantity          InvoiceDate  \
748142   C556445             M        Manual        -1  2011-06-10 15:31:00   
241824   C512770             M        Manual        -1  2010-06-17 16:52:00   
241827    512771             M        Manual         1  2010-06-17 16:53:00   
320581   C520667  BANK CHARGES  Bank Charges        -1  2010-08-27 13:42:00   
1050063  C580605     AMAZONFEE    AMAZON FEE        -1  2011-12-05 11:36:00   

            Price  Customer ID         Country  
748142   38970.00      15098.0  United Kingdom  
241824   25111.09      17399.0  United Kingdom  
241827   25111.09          NaN  United Kingdom  
320581   18910.69          NaN  United Kingdom  
1050063  17836.46          NaN  United Kingdom  
        Invoice StockCode      Description  Quantity          InvoiceDate  \
179403  A506401         B  Adjust bad debt         1  2010-04-29 13:36:00   
276274  A516228         B  Adjust bad debt         1  2010-07-19 11:24:00   


In [60]:
print("Cancelaciones con Customer ID nulo: ", neg_qty_cancelled['Customer ID'].isnull().sum())

print("StockCode Administrativo + Cancelaciones con Customer ID nulo")
print(neg_qty_cancelled[neg_qty_cancelled['Customer ID'].isnull()]['StockCode'].value_counts())

Cancelaciones con Customer ID nulo:  749
StockCode Administrativo + Cancelaciones con Customer ID nulo
StockCode
M               138
S               101
BANK CHARGES     61
POST             48
AMAZONFEE        39
               ... 
21527             1
72802A            1
21102             1
90029             1
22169             1
Name: count, Length: 312, dtype: int64


HALLAZGO: El indicador principal si una orden es por parte de un cliente (Valida / Cancelada) es por medio de un StockCode numerico. Adedmas de un Invoice 'C' con Quantity negativa para cancelaciones validas.<br><br>
Una orden puede tener Quantity negativo pero no clasificarse como canceladas y tener StockCode numerico. Lo que representa un ajuste administrativo.

In [104]:
df[(df['Invoice'].astype(str).str.startswith('C')) & (df['Quantity'] >= 0)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


In [105]:
print(df['Country'].unique())

['United Kingdom' 'France' 'USA' 'Belgium' 'Australia' 'EIRE' 'Germany'
 'Portugal' 'Japan' 'Denmark' 'Nigeria' 'Netherlands' 'Poland' 'Spain'
 'Channel Islands' 'Italy' 'Cyprus' 'Greece' 'Norway' 'Austria' 'Sweden'
 'United Arab Emirates' 'Finland' 'Switzerland' 'Unspecified' 'Malta'
 'Bahrain' 'RSA' 'Bermuda' 'Hong Kong' 'Singapore' 'Thailand' 'Israel'
 'Lithuania' 'West Indies' 'Lebanon' 'Korea' 'Brazil' 'Canada' 'Iceland'
 'Saudi Arabia' 'Czech Republic' 'European Community']


In [63]:
# Todos los StockCode que no siguen el patrón de producto (dígitos, con o sin letra final)
stockcodes_admin = df[~df['StockCode'].str.match(r'^\d{5}[A-Za-z]?$')]['StockCode'].value_counts()
print(stockcodes_admin)


StockCode
POST         2122
DOT          1446
M            1421
15056BL       923
C2            282
             ... 
DCGS0074        1
DCGS0073        1
DCGS0071        1
DCGS0066P       1
DCGS0067        1
Name: count, Length: 68, dtype: int64


In [96]:
print(df[df['StockCode'].str.startswith('DCGS')])
print(df[(df['StockCode'] == '15056BL') & (df['Description'].isna())])
print(df[(df['StockCode'].str.startswith('DCGS')) & (df['Description'].isin(['update', 'ebay']))][['StockCode', 'Description']].drop_duplicates())

        Invoice  StockCode                   Description  Quantity  \
2377     489597   DCGS0058              MISO PRETTY  GUM         1   
2378     489597   DCGS0068             DOGS NIGHT COLLAR         1   
8371     490074   DCGS0004    HAYNES CAMPER SHOULDER BAG         1   
8372     490074   DCGS0058              MISO PRETTY  GUM         1   
8373     490074   DCGS0076  SUNJAR LED NIGHT NIGHT LIGHT         1   
...         ...        ...                           ...       ...   
843773   564825  DCGSSGIRL               GIRLS PARTY BAG         1   
891427   568716   DCGSSBOY                BOYS PARTY BAG         2   
933664   571931  DCGSSGIRL               GIRLS PARTY BAG         1   
1000063  576840  DCGSSGIRL               GIRLS PARTY BAG         1   
1044172  580115   DCGSSBOY                BOYS PARTY BAG         1   

                 InvoiceDate  Price  Customer ID         Country  
2377     2009-12-01 14:28:00   0.83          NaN  United Kingdom  
2378     2009-12-01 14:28

NOTAS FINALES: <br>
Las ordenes de clientes sin cancelacion son aquellas con un StockCode que incluya 4 digitos, Quantity positivo y una Description notnull. <br><br>
Las excepciones erroneas son: <br>
- Description == 'update'
- Description == 'ebay' se descarta al tener Quantity negativo sin contar con 'C' en Invoice <br> <br>

Las excepciones validas son:
- StockCode == 'DCGSSBOY'
- StockCode == 'DCGSSGIRL'

En caso de cancelaciones validas son aquellas que incluyan 'C' en Invoice ademas de un StockCode con 4 digitos.